# PM Skills with Qwen on ModelScope (free quota)

ModelScope API-Inference gives every account a daily free quota for Qwen, DeepSeek, GLM and others, through an OpenAI-compatible endpoint. The account must be linked to an Alibaba Cloud account once (free) before calls work.

## 1. Install

The `pm-skills` package bundles the skills as data; nothing else is needed (the model call below uses only the standard library).

In [ ]:
%pip install -q pm-skills

## 2. Connect

Set `MODELSCOPE_TOKEN` to your ModelScope access token (modelscope.cn → 访问令牌, or modelscope.ai for the international site, which uses `https://api-inference.modelscope.ai/v1`).

In [ ]:
import json, os, urllib.request

def chat(base_url, model, system, user, key=None, timeout=120):
    """One chat completion from any OpenAI-compatible endpoint, standard library only."""
    body = json.dumps({"model": model, "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]}).encode()
    headers = {"content-type": "application/json"}
    if key:
        headers["authorization"] = "Bearer " + key
    req = urllib.request.Request(base_url.rstrip("/") + "/chat/completions", data=body, headers=headers)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.load(r)["choices"][0]["message"]["content"]

BASE_URL = os.environ.get("MODELSCOPE_BASE_URL", "https://api-inference.modelscope.cn/v1")
MODEL = "Qwen/Qwen3.8-27B"
API_KEY = os.environ.get("MODELSCOPE_TOKEN")
assert API_KEY, "Set MODELSCOPE_TOKEN first"

## 3. Find the right skill

`search_skills` ranks skills by how well their name and description match your words.

In [ ]:
import pm_skills

for s in pm_skills.search_skills("weekly report", limit=5):
    print(s["name"], "-", s["title"])

## 4. Run a skill

`skill_prompt` returns the skill's full instructions; send them as the system prompt and your material as the user message.

In [ ]:
notes = """本周完成支付页改版，灰度 20%，转化率提升 1.8%。
第三方支付接口两次超时，已提工单。
下周全量上线，复盘灰度数据。"""
print(chat(BASE_URL, MODEL, pm_skills.skill_prompt("cn-weekly-report"), notes, key=API_KEY))

## 5. Route automatically

Let the search pick the skill from a plain-language request, then run it.

In [ ]:
request = "my landlord kept my deposit for cleaning, what can I do"
best = pm_skills.search_skills("deposit", limit=1)[0]
print("Using:", best["name"])
print(chat(BASE_URL, MODEL, pm_skills.skill_prompt(best["name"]), request, key=API_KEY)[:1500])

Skills: https://github.com/mohitagw15856/pm-claude-skills · The output is a draft for you to check, not professional advice.